# Entrega 3 - Modelo, metricas y preprocesamiento

Este notebook ejecuta el pipeline reproducible de semana 7 usando los modulos en `src/`. No crea formulas analiticas nuevas; compara estructuras de datos para Tableau con metricas de validacion estructural.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT


WindowsPath('d:/Data_Visualization_TF/Data_Visualization_TF')

## 1. Carga de datos

El archivo esperado es `data/raw/all_ai_models.csv`.


In [2]:
from src.config import RAW_DATASET
from src.io_utils import load_csv

raw_df = load_csv(RAW_DATASET)
raw_df.shape


(3509, 57)

## 2. Preprocesamiento reproducible


In [3]:
from src.preprocessing import clean_ai_models, build_quality_summary

clean_df = clean_ai_models(raw_df)
quality_summary = build_quality_summary(raw_df, clean_df)
display(quality_summary)
clean_df.head()


,metric,value
0,raw_rows,3509.00
1,raw_columns,57.00
2,clean_rows,3405.00
3,clean_columns,22.00
4,duplicate_model_rows_after_cleaning,0.00
5,critical_null_rows_after_cleaning,0.00
6,min_year,1950.00
7,max_year,2026.00
8,coverage_pct__Parameters,65.87
9,coverage_pct__Training compute (FLOP),40.12


,Model,Organization,Country (of organization),Publication date,Organization categorization,Domain,Task,Parameters,Training compute (FLOP),Training dataset size (total),...,Hardware quantity,Training hardware,Model accessibility,Training code accessibility,Open model weights?,Citations,Confidence,Notability criteria,Year,Month
0,Gemma 4 31B IT,Google DeepMind,United States of America,2026-04-02,Industry,Language,"Language modeling/generation,Question answering",3.100000e+10,NaN,NaN,...,NaN,NaN,Open weights (restricted use),Unknown,Yes,NaN,Likely,Unknown,2026.0,4.0
1,Nemotron 3 Super,NVIDIA,United States of America,2026-03-11,Industry,Language,"Language modeling/generation,Coding",1.200000e+11,NaN,NaN,...,NaN,NaN,Unknown,Unknown,Unknown,NaN,Likely,Discretionary,2026.0,3.0
2,GPT-5.4,OpenAI,United States of America,2026-03-05,Industry,Multimodal,"Language modeling/generation,Question answering",NaN,NaN,NaN,...,NaN,NaN,API access,Unreleased,No,NaN,Likely,Significant use,2026.0,3.0
3,GPT-5.4 Pro,OpenAI,United States of America,2026-03-05,Industry,Multimodal,"Language modeling/generation,Question answering",NaN,NaN,NaN,...,NaN,NaN,API access,Unreleased,No,NaN,Likely,Discretionary,2026.0,3.0
4,Gemini 3.1 Flash-Lite,Google,United States of America,2026-03-03,Industry,Language,"Language modeling/generation,Code generation",NaN,NaN,NaN,...,NaN,NaN,API access,Unreleased,No,NaN,Likely,Unknown,2026.0,3.0


## 3. Comparacion de opciones de modelo

Se comparan dos opciones: tabla plana y esquema estrella. Las metricas verifican preservacion de filas, unicidad de modelos, nulos criticos y llaves relacionales.


In [4]:
from src.modeling import compare_model_options

model_comparison = compare_model_options(clean_df)
display(model_comparison)


,option,description,tables,fact_rows,unique_model_keys,duplicate_model_rows,critical_null_rows,relationship_null_keys,recommended
0,A_flat_table,Tabla limpia unica para Tableau,1,3405,3405,0,0,0,No
1,B_star_schema,"Tabla de hechos con dimensiones de pais, organ...",5,3405,3405,0,0,0,Si


## 4. Exportacion para Tableau


In [5]:
from src.config import CLEAN_DATASET, OUTPUTS_REPORTS, OUTPUTS_TABLEAU
from src.io_utils import save_csv
from src.modeling import build_flat_model, build_star_schema

save_csv(clean_df, CLEAN_DATASET)
save_csv(quality_summary, OUTPUTS_REPORTS / 'quality_summary.csv')
save_csv(model_comparison, OUTPUTS_REPORTS / 'model_options_comparison.csv')

for name, table in build_flat_model(clean_df).items():
    save_csv(table, OUTPUTS_TABLEAU / f'{name}.csv')

for name, table in build_star_schema(clean_df).items():
    save_csv(table, OUTPUTS_TABLEAU / f'{name}.csv')

print('Archivos exportados en:', OUTPUTS_TABLEAU)


Archivos exportados en: D:\Data_Visualization_TF\Data_Visualization_TF\outputs\tableau


## 5. Decision metodologica

La recomendacion se documenta en `docs/entrega_3_modelo_metricas_preprocesamiento.md`. En terminos operativos, se elige el esquema estrella cuando conserva las filas del dataset limpio, mantiene unicidad por `Model` y no genera llaves relacionales nulas.
